In [1]:
# baseline.py

import numpy as np
import pandas as pd
import random
import json

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    log_loss,
)

from sklearn.calibration import calibration_curve

In [2]:
# -------------------------------
# 1. Reproducibility
# -------------------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

In [3]:
# -------------------------------
# 2. Load Dataset
# -------------------------------
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

feature_names = X.columns.tolist()

In [4]:
# -------------------------------
# 3. Stratified Train-Test Split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

In [5]:
# -------------------------------
# 4. Preprocessing (No Leakage)
# -------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [6]:
# -------------------------------
# 5. Train Models
# -------------------------------
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=5000, random_state=SEED
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=200, random_state=SEED
    ),
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)
    loss = log_loss(y_test, y_prob)

    # Calibration curve
    prob_true, prob_pred = calibration_curve(y_test, y_prob, n_bins=10)

    results[name] = {
        "accuracy": acc,
        "f1_score": f1,
        "roc_auc": roc,
        "log_loss": loss,
        "calibration_true": prob_true.tolist(),
        "calibration_pred": prob_pred.tolist(),
    }

    print(f"\nModel: {name}")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"ROC AUC: {roc:.4f}")
    print(f"Log Loss: {loss:.4f}")

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


Model: LogisticRegression
Accuracy: 0.9825
F1 Score: 0.9861
ROC AUC: 0.9954
Log Loss: 0.0777
Confusion Matrix:
[[41  1]
 [ 1 71]]

Model: RandomForest
Accuracy: 0.9561
F1 Score: 0.9655
ROC AUC: 0.9932
Log Loss: 0.1118
Confusion Matrix:
[[39  3]
 [ 2 70]]


In [7]:
# -------------------------------
# 6. Save Baseline Feature Stats
# -------------------------------
baseline_feature_stats = {
    "train_mean": X_train.mean().to_dict(),
    "train_std": X_train.std().to_dict(),
    "test_mean": X_test.mean().to_dict(),
    "test_std": X_test.std().to_dict(),
}

with open("baseline_feature_stats.json", "w") as f:
    json.dump(baseline_feature_stats, f, indent=4)

with open("baseline_metrics.json", "w") as f:
    json.dump(results, f, indent=4)


print("\nBaseline experiment completed successfully.")


Baseline experiment completed successfully.


In [8]:
import numpy as np

def compute_ece(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    binids = np.digitize(y_prob, bins) - 1
    ece = 0.0

    for i in range(n_bins):
        mask = binids == i
        if np.any(mask):
            bin_acc = np.mean(y_true[mask])
            bin_conf = np.mean(y_prob[mask])
            ece += np.abs(bin_acc - bin_conf) * np.sum(mask) / len(y_prob)

    return ece


In [9]:
ece = compute_ece(y_test.values, y_prob)
print(f"ECE: {ece:.4f}")



ECE: 0.0531


# Save Feature Importances

In [10]:
#Logistic_regression
if name == "LogisticRegression":
    coef_dict = dict(zip(feature_names, model.coef_[0]))
    results[name]["coefficients"] = coef_dict


In [11]:
#Random_forest
if name == "RandomForest":
    importance_dict = dict(zip(feature_names, model.feature_importances_))
    results[name]["feature_importances"] = importance_dict


# Save Prediction Confidence Distribution

In [12]:
results[name]["prediction_confidence_mean"] = float(np.mean(y_prob))
results[name]["prediction_confidence_std"] = float(np.std(y_prob))


# Record Class Distribution Explicitly

In [13]:
results[name]["train_class_distribution"] = y_train.value_counts(normalize=True).to_dict()
results[name]["test_class_distribution"] = y_test.value_counts(normalize=True).to_dict()


In [14]:
results

{'LogisticRegression': {'accuracy': 0.9824561403508771,
  'f1_score': 0.9861111111111112,
  'roc_auc': np.float64(0.9953703703703703),
  'log_loss': 0.07774649384096739,
  'calibration_true': [0.0, 0.0, 0.5, 1.0, 1.0, 1.0, 1.0, 0.9838709677419355],
  'calibration_pred': [0.0028053752262647896,
   0.16176820612364742,
   0.3588710592240636,
   0.5390642584762925,
   0.6212626093643917,
   0.707218502217024,
   0.8626934478517407,
   0.9832463240903977]},
 'RandomForest': {'accuracy': 0.956140350877193,
  'f1_score': 0.9655172413793104,
  'roc_auc': np.float64(0.9932208994708995),
  'log_loss': 0.11178520539161924,
  'calibration_true': [0.0, 0.0, 0.5, 0.5, 1.0, 0.5, 1.0, 1.0, 1.0],
  'calibration_pred': [0.01147058823529412,
   0.11166666666666665,
   0.275,
   0.345,
   0.55,
   0.6583333333333333,
   0.7275,
   0.8538888888888889,
   0.9841666666666666],
  'feature_importances': {'mean radius': np.float64(0.05866225824281181),
   'mean texture': np.float64(0.014316720485671626),
   'm